In [0]:
from pyspark.sql.functions import col, when, floor

# --- BIỂU ĐỒ 1: Phân bổ rủi ro theo Grade (Dataset 1) ---
# Đọc từ bảng 'loan_results' (bảng đã qua model xử lý)
df_loan_res = spark.read.format("delta").load("/Volumes/workspace/default/data_csv/loan_results")

risk_by_grade = df_loan_res.groupBy("grade", "prediction").count()
risk_by_grade.write.format("delta").mode("overwrite").saveAsTable("default.viz_loan_grade")

# --- BIỂU ĐỒ 2: DTI vs Loan Amount (Scatter Plot) ---
dti_amt = df_loan_res.select("dti", "loan_amnt", "prediction").limit(1000)
dti_amt.write.format("delta").mode("overwrite").saveAsTable("default.viz_loan_scatter")

# --- BIỂU ĐỒ 3: Chốt đơn theo Nghề nghiệp (Dataset 2) ---
# Đọc từ bảng 'marketing_results'
df_mkt_res = spark.read.format("delta").load("/Volumes/workspace/default/data_csv/marketing_results")

# Lưu ý: Ở Mkt, cột nhãn của mày là 'label' (đã mã hóa từ yes/no)
job_conv = df_mkt_res.groupBy("job", "prediction").count() 
job_conv.write.format("delta").mode("overwrite").saveAsTable("default.viz_mkt_job")

# --- BIỂU ĐỒ 4: Phân bổ Số dư (Balance) ---
balance_dist = df_mkt_res.withColumn("balance_range", (floor(col("balance") / 5000) * 5000).cast("string")) \
    .groupBy("balance_range", "prediction").count()
balance_dist.write.format("delta").mode("overwrite").saveAsTable("default.viz_mkt_balance")

print("✅ Đã chuẩn bị xong 4 bảng dữ liệu cho Dashboard! Bây giờ Backend có thể query cột 'prediction' rồi.")



In [0]:
# --- THỐNG KÊ 5: Tình trạng nhà ở của người đi vay (Dataset 1 - Thực tế) ---
# Dùng để biết đối tượng nào hay đi vay nhất
home_stats = spark.read.format("delta").load("/Volumes/workspace/default/data_csv/loan_silver") \
    .groupBy("home_ownership").count()
home_stats.write.format("delta").mode("overwrite").saveAsTable("default.viz_loan_home")

# --- THỐNG KÊ 6: Tỷ lệ chốt đơn thật theo Học vấn (Dataset 2 - Thực tế) ---
# Dùng cột 'y' (yes/no) gốc trong dataset
edu_stats = spark.read.format("delta").load("/Volumes/workspace/default/data_csv/marketing_silver") \
    .groupBy("education", "y").count()
edu_stats.write.format("delta").mode("overwrite").saveAsTable("default.viz_mkt_edu")

print("✅ Đã tạo thêm 2 bảng thống kê thực tế!")